In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

def prepare():
    module_path = os.path.abspath(os.path.join('..'))
    if module_path not in sys.path:
        sys.path.append(module_path)

In [ ]:
import torch
import numpy as np
prepare()
from exp_labelcert_collective import run

In [ ]:
model_params = dict(
    label = "GCN", 
    model = "GCN", 
    normalization = "row_normalization",
    activation = "relu",
    depth = 1,
    regularizer = 0.001,
    pred_method = "svm",
    bias = False,
    alpha_tol = 1e-4,
    solver = "qplayer",
)

certificate_params = dict(
    delta = 0.01,
    TimeLimit = 86400,
    LogToConsole = 1,
    OutputFlag = 1,
    Threads = 2,
    Presolve = 2
)

verbosity_params = dict(
    debug_lvl = "warning"
)  

other_params = dict(
    device = "0",
    dtype = torch.float64,
    allow_tf32 = False,
    path_gurobi_license = "path/to/your/gurobi/license"
)

In [ ]:
data_params = dict(
    dataset = "cba",
    learning_setting = "transductive", 
    specification = dict(
        classes = 2,
        n_trn_labeled = 10,
        n_trn_unlabeled = 0,
        n_val = 10,
        n_test = 180,
        sigma = 1,
        avg_within_class_degree = 1.58 * 2,
        avg_between_class_degree = 0.37 * 2,
        K = 1.5,
        m = 2,
        seed = 0 # used to generate the dataset & data split
    )
)

In [ ]:
import pandas as pd
import time

seeds = [0, 1, 2, 3, 4]
delta = 0.3
certificate_params["delta"] = delta

metrics = [
    "accuracy_test",
    "accuracy_trn",
    "accuracy_cert_pois_robust",
    "accuracy_cert_pois_unrobust"
]

summary = []

for seed in seeds:
    data_params["specification"]["seed"] = seed
    
    start_time = time.time()
    result = run(data_params, model_params, certificate_params, verbosity_params, other_params, seed)
    end_time = time.time()
    runtime = round(end_time - start_time, 2)
    
    summary.append({k: result[k] for k in metrics} | {"delta": delta} | {"runtime": runtime})
    
    
df = pd.DataFrame(summary, index=[f"Seed {s}" for s in seeds])
df.index.name = "seed"
df.to_csv(f'results/collective/cba-{delta:.2f}.csv', index=True)
df

In [ ]:
#  Complexity Cascade Data Generation
# dense epsilon sweep for the GCN model on the CBA dataset
import os
import sys
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path


module_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    print(f"Added '{module_path}' to sys.path to find project modules.")

from exp_labelcert_collective import run

model_params = {
    'label': 'GCN',
    'model': 'GCN',
    'normalization': 'sym_normalization',  
    'activation': 'relu',
    'depth': 1,
    'regularizer': 0.001,
    'pred_method': 'svm',
    'bias': False,
    'alpha_tol': 1e-4,
    'solver': 'qplayer'
}

data_params = {
    'dataset': 'cba',  
    'learning_setting': 'transductive',
    'specification': {
        'classes': 2,
        'n_trn_labeled': 10,
        'n_trn_unlabeled': 0,
        'n_val': 10,
        'n_test': 180,
        'sigma': 1,
        'avg_within_class_degree': 3.16,  
        'avg_between_class_degree': 0.74  
    }
}

other_params = {
    'device': 'cpu',  
    'dtype': torch.float64,
    'allow_tf32': False,
    'path_gurobi_license': '/mnt/c/Users/emiel/gurobi.lic' 
}

certificate_base_params = {
    'TimeLimit': 86400,
    'LogToConsole': 1,
    'OutputFlag': 1,
    'Threads': 2,
    'Presolve': 2
}

verbosity_params = {'debug_lvl': 'warning'}


eps_coarse = np.linspace(0.00, 0.30, 16).tolist()
eps_fine = np.linspace(0.13, 0.18, 26).tolist()
epsilons = sorted(set(eps_coarse + eps_fine))

additional_eps = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.5, 1.0]
epsilons = sorted(set(epsilons + additional_eps))

seeds = range(10)  

log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)
output_filename = "final_gcn_on_cba_collective_data.csv"
output_path = log_dir / output_filename


print("--- Starting Final Data Generation for GCN on CBA (Collective) ---")
print(f"Running {len(epsilons)} epsilon points across {len(seeds)} seeds...")
print(f"Total experiments: {len(epsilons) * len(seeds)}")
all_results = []
start_time_total = time.time()

for seed_val in seeds:
    print(f"\nProcessing Seed: {seed_val}/{len(seeds)-1}...")
    data_params['specification']['seed'] = int(seed_val)
    
    for i, eps in enumerate(epsilons):
        certificate_params = certificate_base_params.copy()
        certificate_params['delta'] = float(eps)
        
        try:
            out = run(
                data_params=data_params,
                model_params=model_params,
                certificate_params=certificate_params,
                verbosity_params=verbosity_params,
                other_params=other_params,
                seed=int(seed_val)
            )
            
            out.update({
                'delta': float(eps), 
                'seed': int(seed_val),
                'dataset': 'cba',
                'model': 'GCN'
            })
            all_results.append(out)
            
            robust_ratio = out.get('accuracy_cert_pois_robust', 'N/A')
            gurobi_nodes = out.get('gurobi_node_count', 'N/A')
            print(f"  ({i+1}/{len(epsilons)}) ε={eps:.4f} | Robust Ratio: {robust_ratio} | Gurobi Nodes: {gurobi_nodes}")
            
        except Exception as e:
            print(f"  ERROR at ε={eps:.4f}, seed={seed_val}: {str(e)}")
            error_result = {
                'delta': float(eps),
                'seed': int(seed_val),
                'dataset': 'cba',
                'model': 'GCN',
                'error': str(e),
                'accuracy_cert_pois_robust': np.nan,
                'gurobi_node_count': np.nan
            }
            all_results.append(error_result)

df_final = pd.DataFrame(all_results)
df_final.to_csv(output_path, index=False)

end_time_total = time.time()
print(f"\n--- ✅ Experiment Complete ---")
print(f"Total runtime: {(end_time_total - start_time_total) / 60:.2f} minutes")
print(f"Final data for {len(df_final)} runs saved to '{output_path}'")
print(f"Success rate: {len(df_final[~df_final.get('error', pd.Series()).notna()])} / {len(df_final)}")

if len(df_final) > 0:
    print(f"\n--- Quick Summary ---")
    valid_results = df_final[~df_final.get('error', pd.Series()).notna()]
    if len(valid_results) > 0:
        print(f"Valid results: {len(valid_results)}")
        if 'accuracy_cert_pois_robust' in valid_results.columns:
            print(f"Robust ratio range: {valid_results['accuracy_cert_pois_robust'].min():.4f} - {valid_results['accuracy_cert_pois_robust'].max():.4f}")
        if 'gurobi_node_count' in valid_results.columns:
            print(f"Gurobi nodes range: {valid_results['gurobi_node_count'].min()} - {valid_results['gurobi_node_count'].max()}")